# Apache Iceberg Small Dataset Generator & Loader (MinIO + PySpark)

This notebook generates a lightweight, tutorial-friendly E-commerce dataset directly onto your Docker **MinIO S3** bucket (`s3a://warehouse/raw_data/`):
- **Customers (`customers_csv`)**: 50 rows in a single CSV file.
- **Orders (`orders_phase1_parquet`)**: 1,000 rows in a single Parquet file spanning **3 distinct days across 3 different months** (`2026-07-15`, `2026-08-20`, `2026-09-10`).

Having dates across different months makes it easy to demonstrate **Hidden Partitioning**, **Partition Evolution** (e.g., `days()` vs `months()`), `MERGE INTO`, Time Travel, and WAP branching with instant execution.

### Prerequisites
- Run inside your `spark-iceberg` Jupyter Notebook (`http://localhost:8888`).
- `spark-defaults.conf` must have your MinIO S3A credentials and Iceberg `demo` catalog configured.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Inherits MinIO S3A and Iceberg JDBC catalog configs from spark-defaults.conf
spark = SparkSession.builder.appName('Generate-Small-Ecommerce-Datasets').getOrCreate()

# --- Dataset Configuration ---
NUM_CUSTOMERS = 50
NUM_ORDERS = 1_000

# 3 specific dates across 3 different months (July, August, September 2026)
ORDER_DATES = ['2026-07-15', '2026-08-20', '2026-09-10']

CUSTOMERS_CSV_PATH = 's3a://warehouse/raw_data/customers_csv'
ORDERS_PARQUET_PATH = 's3a://warehouse/raw_data/orders_phase1_parquet'

print('Spark Session initialized successfully.')

Spark Session initialized successfully.


26/11/24 01:44:26 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 1. Create Namespace & Iceberg Tables

- `demo.ecommerce.customers`: Unpartitioned dimension table.
- `demo.ecommerce.orders`: Fact table with Iceberg **Hidden Partitioning** (`days(order_timestamp)`) and **Merge-on-Read (MoR)** properties, ready for schema and partition evolution.

In [2]:
spark.sql('CREATE NAMESPACE IF NOT EXISTS demo.ecommerce')

# Optional: Uncomment the two lines below if you want a completely fresh reset before re-running
# spark.sql('DROP TABLE IF EXISTS demo.ecommerce.customers')
# spark.sql('DROP TABLE IF EXISTS demo.ecommerce.orders')

spark.sql('''
CREATE TABLE IF NOT EXISTS demo.ecommerce.customers (
    customer_id BIGINT,
    customer_name STRING,
    segment STRING,
    signup_date DATE
)
USING iceberg
TBLPROPERTIES (
    'format-version' = '2',
    'write.parquet.compression-codec' = 'zstd'
)
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS demo.ecommerce.orders (
    order_id BIGINT,
    customer_id BIGINT,
    order_timestamp TIMESTAMP,
    status STRING,
    total_amount DECIMAL(10, 2)
)
USING iceberg
PARTITIONED BY (days(order_timestamp))
TBLPROPERTIES (
    'format-version' = '2',
    'write.update.mode' = 'merge-on-read',
    'write.delete.mode' = 'merge-on-read',
    'write.merge.mode' = 'merge-on-read',
    'write.parquet.compression-codec' = 'zstd'
)
''')

print('Namespace and Iceberg tables are ready.')

Namespace and Iceberg tables are ready.


## 2. Generate Customers Dataset (50 Rows, Single CSV File)

Generates 50 customer records (`customer_id` 1 to 50) and writes a single `.csv` file with headers to MinIO.

In [3]:
segments = F.array(
    F.lit('Retail'), F.lit('Wholesale'), F.lit('Enterprise'), F.lit('SMB')
)

customers_df = (
    spark.range(1, NUM_CUSTOMERS + 1, numPartitions=1)
    .select(
        F.col('id').cast('bigint').alias('customer_id'),
        F.concat(F.lit('Customer_'), F.col('id')).alias('customer_name'),
        F.element_at(segments, (F.floor(F.rand(10) * 4) + 1).cast('int')).alias('segment'),
        F.date_add(
            F.lit('2024-01-01').cast('date'),
            F.floor(F.rand(20) * 700).cast('int')
        ).alias('signup_date'),
    )
)

(
    customers_df
    .coalesce(1)
    .write
    .mode('overwrite')
    .option('header', 'true')
    .csv(CUSTOMERS_CSV_PATH)
)

print(f'Customers CSV (50 rows) written to: {CUSTOMERS_CSV_PATH}')
customers_df.show(10, truncate=False)

26/11/24 01:44:40 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/11/24 01:44:40 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/11/24 01:44:41 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


Customers CSV (50 rows) written to: s3a://warehouse/raw_data/customers_csv
+-----------+-------------+----------+-----------+
|customer_id|customer_name|segment   |signup_date|
+-----------+-------------+----------+-----------+
|1          |Customer_1   |Retail    |2025-02-23 |
|2          |Customer_2   |SMB       |2025-01-30 |
|3          |Customer_3   |Enterprise|2025-10-19 |
|4          |Customer_4   |SMB       |2024-05-05 |
|5          |Customer_5   |Retail    |2024-06-03 |
|6          |Customer_6   |Wholesale |2025-03-30 |
|7          |Customer_7   |SMB       |2024-03-29 |
|8          |Customer_8   |Enterprise|2025-02-22 |
|9          |Customer_9   |Enterprise|2024-02-01 |
|10         |Customer_10  |Wholesale |2024-01-08 |
+-----------+-------------+----------+-----------+
only showing top 10 rows



## 3. Generate Orders Dataset (1,000 Rows, Single Parquet File Across 3 Days in Different Months)

Generates 1,000 order records (`order_id` 1001 to 2000) referencing `customer_id` 1 to 50, distributed across **2026-07-15**, **2026-08-20**, and **2026-09-10** with random timestamps within each day.

In [4]:
statuses = F.array(
    F.lit('PENDING'), F.lit('PROCESSING'), F.lit('COMPLETED'), F.lit('CANCELLED')
)

date_choices = F.array(*[F.lit(d) for d in ORDER_DATES])
num_dates = len(ORDER_DATES)

orders_df = (
    spark.range(1001, 1001 + NUM_ORDERS, numPartitions=2)
    .withColumn('chosen_date', F.element_at(date_choices, (F.floor(F.rand(2) * num_dates) + 1).cast('int')))
    .withColumn('rand_seconds', F.floor(F.rand(5) * 86400).cast('bigint'))
    .select(
        F.col('id').cast('bigint').alias('order_id'),
        (F.floor(F.rand(1) * NUM_CUSTOMERS) + 1).cast('bigint').alias('customer_id'),
        F.timestamp_seconds(
            F.unix_timestamp(F.col('chosen_date'), 'yyyy-MM-dd') + F.col('rand_seconds')
        ).alias('order_timestamp'),
        F.element_at(statuses, (F.floor(F.rand(3) * 4) + 1).cast('int')).alias('status'),
        F.round(F.lit(25.0) + (F.rand(4) * 1975.0), 2)
         .cast('decimal(10, 2)')
         .alias('total_amount'),
    )
)

(
    orders_df
    .coalesce(1)
    .write
    .mode('overwrite')
    .option('compression', 'snappy')
    .parquet(ORDERS_PARQUET_PATH)
)

print(f'Orders Parquet (1,000 rows) written to: {ORDERS_PARQUET_PATH}')
orders_df.show(10, truncate=False)

Orders Parquet (1,000 rows) written to: s3a://warehouse/raw_data/orders_phase1_parquet
+--------+-----------+-------------------+----------+------------+
|order_id|customer_id|order_timestamp    |status    |total_amount|
+--------+-----------+-------------------+----------+------------+
|1001    |32         |2026-08-20 00:34:25|PROCESSING|1907.46     |
|1002    |30         |2026-07-15 02:16:56|COMPLETED |155.62      |
|1003    |7          |2026-08-20 18:29:31|CANCELLED |310.45      |
|1004    |4          |2026-08-20 13:04:05|COMPLETED |867.14      |
|1005    |43         |2026-09-10 08:05:13|PENDING   |35.98       |
|1006    |36         |2026-08-20 00:49:12|PENDING   |977.80      |
|1007    |13         |2026-08-20 06:24:18|CANCELLED |816.60      |
|1008    |7          |2026-08-20 22:37:36|CANCELLED |1835.04     |
|1009    |20         |2026-09-10 06:46:37|PROCESSING|210.86      |
|1010    |31         |2026-07-15 01:56:32|CANCELLED |1273.49     |
+--------+-----------+-------------------+

## 4. Verify Generated Files & Date/Month Distribution

Confirm exact row counts, physical file sizes on MinIO, and that all 1,000 orders land cleanly across the 3 target dates in 3 different months.

In [5]:
jvm = spark._jvm
hconf = spark._jsc.hadoopConfiguration()

for label, s3_path in [('Customers CSV', CUSTOMERS_CSV_PATH), ('Orders Parquet', ORDERS_PARQUET_PATH)]:
    path = jvm.org.apache.hadoop.fs.Path(s3_path)
    fs = path.getFileSystem(hconf)
    for status in fs.listStatus(path):
        name = status.getPath().getName()
        if name.endswith(('.parquet', '.csv')):
            size_kb = status.getLen() / 1024
            print(f'{label} -> {name} | Size: {size_kb:.2f} KB')

print('\n--- Orders Count by Date (3 Days Across 3 Months) ---')
(
    spark.read.parquet(ORDERS_PARQUET_PATH)
    .groupBy(F.to_date('order_timestamp').alias('order_date'))
    .count()
    .orderBy('order_date')
    .show()
)

Customers CSV -> part-00000-c6650382-3787-46c0-9861-ebb0a9d56790-c000.csv | Size: 1.71 KB
Orders Parquet -> part-00000-704d514d-eb4a-4532-bdde-564b3490bdfc-c000.snappy.parquet | Size: 19.98 KB

--- Orders Count by Date (3 Days Across 3 Months) ---
+----------+-----+
|order_date|count|
+----------+-----+
|2026-07-15|  318|
|2026-08-20|  324|
|2026-09-10|  358|
+----------+-----+



## 5. Load Generated Files into Iceberg Tables

- Reads the CSV with explicit type casting and appends to `demo.ecommerce.customers`.
- Reads the Parquet file and appends to `demo.ecommerce.orders`; Iceberg automatically splits the 1,000 rows into 3 hidden daily partitions (`2026-07-15`, `2026-08-20`, `2026-09-10`).

In [6]:
# 1. Load Customers CSV into Iceberg
cust_in = (
    spark.read.option('header', 'true').csv(CUSTOMERS_CSV_PATH)
    .select(
        F.col('customer_id').cast('bigint'),
        F.col('customer_name').cast('string'),
        F.col('segment').cast('string'),
        F.col('signup_date').cast('date'),
    )
)
cust_in.writeTo('demo.ecommerce.customers').append()
print('Loaded 50 rows into demo.ecommerce.customers.')

# 2. Load Orders Parquet into Iceberg
orders_in = spark.read.parquet(ORDERS_PARQUET_PATH)
orders_in.writeTo('demo.ecommerce.orders').append()
print('Loaded 1,000 rows into demo.ecommerce.orders.')

Loaded 50 rows into demo.ecommerce.customers.
Loaded 1,000 rows into demo.ecommerce.orders.


## 6. Verify Iceberg Metadata, Hidden Partitions & Join

Inspect `.files` to confirm 3 separate daily partition files were created automatically, and run a sample join between `orders` and `customers`.

In [7]:
print('--- Iceberg Orders Physical Files & Hidden Partitions ---')
spark.sql('''
    SELECT file_path, spec_id, partition, record_count, ROUND(file_size_in_bytes / 1024, 2) AS size_kb
    FROM demo.ecommerce.orders.files
''').show(truncate=False)

print('--- Sample Join: Revenue by Order Date & Customer Segment ---')
spark.sql('''
    SELECT
        DATE(o.order_timestamp) AS order_date,
        c.segment,
        COUNT(o.order_id) AS total_orders,
        ROUND(SUM(o.total_amount), 2) AS total_revenue
    FROM demo.ecommerce.orders o
    JOIN demo.ecommerce.customers c
        ON o.customer_id = c.customer_id
    GROUP BY DATE(o.order_timestamp), c.segment
    ORDER BY order_date, c.segment
''').show()

--- Iceberg Orders Physical Files & Hidden Partitions ---
+----------------------------------------------------------------------------------------------------------------------------------+-------+------------+------------+-------+
|file_path                                                                                                                         |spec_id|partition   |record_count|size_kb|
+----------------------------------------------------------------------------------------------------------------------------------+-------+------------+------------+-------+
|s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-07-15/00000-11-7ed76c1c-42ea-4084-bd4a-0f5c9a538d0c-0-00001.parquet|0      |{2026-07-15}|318         |5.12   |
|s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-08-20/00000-11-7ed76c1c-42ea-4084-bd4a-0f5c9a538d0c-0-00003.parquet|0      |{2026-08-20}|324         |5.19   |
|s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-09-